# Lazy Browse — DestinE Climate DT Hourly Portfolio

This notebook explores the **hourly (`clte`) stream** of the Climate DT Generation 2 data portfolio.

**Prerequisites:** run `01_key_destine_once.ipynb` once to authenticate.

In [ ]:
import logging, warnings
import earthkit.data

# Disable earthkit disk cache (polytope_zarr caches decoded arrays in memory)
earthkit.data.config.set("cache-policy", "off")

# Silence verbose output from polytope / earthkit internals
for _ln in ("polytope", "polytope.api", "earthkit.data", "urllib3"):
    logging.getLogger(_ln).setLevel(logging.WARNING)
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [ ]:
from polytope_zarr import PolytopeZarrStore

## 1. Create hourly store

The function `PolytopeZarrStore.from_climate_dt(frequency="hourly")`
automatically selects `PORTFOLIO_GEN2_CLTE`.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────

# ── Choose a levtype (uncomment one) ──────────────────────────────
LEVTYPE = "sfc"                    # 34 vars — surface (14 instant + 20 hourly mean)
# LEVTYPE = "pl"                   #  9 vars — pressure levels (19 levels)
# LEVTYPE = "hl"                   #  2 vars — height levels (100 m, IFS-only)
# LEVTYPE = "sol"                  #  2 vars — soil / snow
# LEVTYPE = "o2d"                  # 12 vars — 2-D ocean & sea ice (daily)
# LEVTYPE = "o3d"                  #  5 vars — 3-D ocean (daily, up to 75 levels)

store = PolytopeZarrStore.from_climate_dt(
    models=["ICON", "IFS-FESOM", "IFS-NEMO"],
    experiment="hist",
    levtype=LEVTYPE,
    frequency="hourly",
    start_date="1990-01-01T00:00:00",
    end_date="2014-12-31T23:00:00",
)
print(store)

## 2. Open as xarray Dataset — lazy browse

In [ ]:
ds = store.open()
ds

## 3. Plot a single hourly field (triggers lazy fetch)

Only now does the store actually call Polytope — fetching data for the
selected model, time, and variable.

In [ ]:
import healpy as hp
import matplotlib.pyplot as plt

field = ds["2t"].sel(model="IFS-FESOM", time="2014-01-01T12:00")
print(f"Fetching {dict(field.sizes)} values...")

hp.mollview(field.values, title="IFS-FESOM — 2m temperature — 2014-01-01 12:00",
            unit="K", cmap="RdYlBu_r", nest=True, flip='geo')
plt.show()

In [ ]:
#annual mean, can take around 6-10 minutes depending on your connection
field2 = ds["avg_tprate"].polytope.sel(model="IFS-FESOM", time=slice("2014-01-01T00:00", "2014-12-31T23:00")).mean("time")

hp.mollview(field2.values,
            title="IFS-FESOM — avg total precipitation rate from hourly values — year 2014",
            unit="m", cmap="Blues", min=0, max=0.0001, nest=True, flip='geo')
plt.show()

In [ ]:
# in the next call, the data is read from cache, and we can apply a different reduction (std instead of mean) without re-reading the data
field2 = ds["avg_tprate"].sel(model="IFS-FESOM", time=slice("2014-01-01T00:00", "2014-12-31T23:00")).std("time")

hp.mollview(field2.values,
            title="IFS-FESOM — annual std of avg total precipitation rate from hourly values — year 2014",
            unit="m", cmap="Blues", min=0, max=0.0001, nest=True, flip='geo')
plt.show()

# Create a daily ocean store

In [ ]:
# ── Configuration ─────────────────────────────────────────────────
LEVTYPE = "o2d"

daily_store = PolytopeZarrStore.from_climate_dt(
    models=["ICON", "IFS-FESOM", "IFS-NEMO"],
    experiment="hist",
    levtype=LEVTYPE,
    frequency="hourly",
    start_date="1990-01-01",
    end_date="2014-12-31",
)
print(daily_store)

In [ ]:
ds = daily_store.open()
ds

In [ ]:
import healpy as hp
import matplotlib.pyplot as plt

ocefield = ds["avg_zos"].sel(model="IFS-FESOM", time="2014-01-01")
print(f"Fetching {dict(ocefield.sizes)} values...")

hp.mollview(ocefield.values, title="IFS-FESOM — Sea Surface Height — 2014-01-01",
            unit="m", cmap="RdYlBu_r", nest=True, flip='geo')
plt.show()

## Notes

- **Automatic batching:** use `.polytope.sel(time=slice(...))` to auto-batch Polytope requests over the requested date range.
- **SFC variable names:** instantaneous fields use standard ECMWF shortNames (`2t`, `sp`, `10si`), while hourly-mean fluxes keep the `avg_` prefix (`avg_tprate`, `avg_ishf`).
- **Ocean/ice (o2d, o3d):** daily resolution, all variables keep `avg_` prefix.
- `store.clear_cache()` frees memory from previously fetched fields.

In [ ]:
store.clear_cache()